In [0]:
# =============================================================================
# NB18 — Activity re-tests with anion-aware descriptors (X_material_v2)
#        LAST exploratory notebook — consolidation follows.
# =============================================================================
# Why: NB08–NB14 used X_material, built WITHOUT anions (NB02b). NB17 showed
# anion-aware descriptors (X_material_v2) are far better for the band gap
# (R² 0.13 → 0.46, same featuriser). The two activity conclusions that depend
# on composition descriptors are re-tested here, with NO other change:
#
#   Part 1 (re-test of NB11/NB12, across papers):
#     Does composition add to experimental conditions?
#     XGBoost, 5×5 repeated GroupKFold by DOI (same seeds as NB11/NB12),
#     paired ΔR² vs EXP_ONLY.
#       EXP_ONLY          experimental variables (X_exp_v2)
#       EXP+V1            + old X_material (reference, reproduces NB12)
#       EXP+V2            + X_material_v2
#       EXP+V2_ANIONS     + anion fractions only (v2_frac_*) — 5 features
#       V2_ONLY           X_material_v2 alone
#
#   Part 2 (re-test of NB14, within papers, strict blocks, base_compound key):
#     Does composition predict which material is better once the design
#     heuristics are neutralised?  Features = measured Eg + X_material_v2.
#     XGBoost OOF (5 repeats) by pair type; permutation test (60) on
#     non_mixed and equal_complexity pairs.
#
# Inputs:  master, X_material, X_material_v2 (NB17), X_exp_v2
# Outputs: ml_perovskites.activity_retest_nb18
# Rules:   encoding/imputation inside folds; DOI never in train and val; SEED=42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import re, subprocess, sys
import numpy as np
import pandas as pd
from sklearn.impute  import SimpleImputer
from sklearn.metrics import r2_score
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS, TE_M = 42, "ml_perovskites", 5, 5, 10
PAIR_MIN, N_PERM = 0.2, 60
RUN_PART1 = True   # set False to skip the across-paper re-test (already run)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

def xgb_model(depth=4, n=400):
    return xgb.XGBRegressor(n_estimators=n, learning_rate=0.05, max_depth=depth,
                            subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                            reg_lambda=1.0, tree_method="hist", random_state=SEED, verbosity=0)

# ---------------------------------------------------------------------------
# Load (conventions of NB11)
# ---------------------------------------------------------------------------
master = spark.table(f"{SCHEMA}.master").toPandas()
X_v1 = spark.table(f"{SCHEMA}.X_material").toPandas()
X_v2 = spark.table(f"{SCHEMA}.X_material_v2").toPandas()
X_ex = spark.table(f"{SCHEMA}.X_exp_v2").toPandas()
for t in [master, X_v1, X_v2, X_ex]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}
for t in [X_v1, X_v2]:
    for c in t.columns:
        if c != "source_row_id":
            t[c] = pd.to_numeric(t[c], errors="coerce")
v1 = [c for c in X_v1.columns if c not in META and X_v1[c].isna().mean() < 0.50]
v2 = [c for c in X_v2.columns if c not in META and X_v2[c].isna().mean() < 0.50]
v2_anion = [c for c in v2 if c.startswith("v2_frac_")]
CAT = [c for c in ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type",
                   "prep_class"] if c in X_ex.columns]
exp_num = [c for c in X_ex.columns if c not in ["source_row_id"] + CAT]
for c in exp_num:
    X_ex[c] = pd.to_numeric(X_ex[c], errors="coerce")
exp = exp_num + CAT

def is_use_h2(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])

master["y"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
base = master[is_use_h2(master["use_h2"]) & master["y"].notna() &
              master["doi_key"].notna()].copy()
df = (base[["source_row_id", "doi_key", "y"]]
      .merge(X_v1[["source_row_id"] + v1], on="source_row_id", how="left")
      .merge(X_v2[["source_row_id"] + v2], on="source_row_id", how="left")
      .merge(X_ex[["source_row_id"] + exp], on="source_row_id", how="left"))
print(f"Activity rows: {len(df)} | DOIs {df['doi_key'].nunique()} | v1 {len(v1)}, "
      f"v2 {len(v2)} (anion fractions {len(v2_anion)}), exp {len(exp)}")
print(f"  rows with v2 descriptors: {int(df[v2[0]].notna().sum())}")

# ===========================================================================
# PART 1 — across papers (re-test of NB11/NB12)
# ===========================================================================
CONFIGS = {"EXP_ONLY": exp, "EXP+V1": exp + v1, "EXP+V2": exp + v2,
           "EXP+V2_ANIONS": exp + v2_anion, "V2_ONLY": v2}
dois = np.array(sorted(df["doi_key"].unique()))

def encode(tr, va, cols):
    Xtr, Xva = tr[cols].copy(), va[cols].copy()
    for cc in [c for c in CAT if c in cols]:
        gm = tr["y"].mean()
        st = tr.groupby(cc)["y"].agg(["mean", "count"])
        enc = ((st["mean"] * st["count"] + gm * TE_M) / (st["count"] + TE_M)).to_dict()
        Xtr[cc], Xva[cc] = tr[cc].map(enc).fillna(gm), va[cc].map(enc).fillna(gm)
    return Xtr.astype(float), Xva.astype(float)

rows = []
for rep in range(N_REPEATS if RUN_PART1 else 0):
    perm = np.random.default_rng(SEED + rep).permutation(dois)     # same as NB11/NB12
    f_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    f = df["doi_key"].map(f_of).values
    for k in range(N_FOLDS):
        tr, va = df[f != k], df[f == k]
        for name, cols in CONFIGS.items():
            Xtr, Xva = encode(tr, va, cols)
            imp = SimpleImputer(strategy="median").fit(Xtr.values)
            p = xgb_model().fit(imp.transform(Xtr.values), tr["y"].values) \
                           .predict(imp.transform(Xva.values))
            rows.append({"part": 1, "config": name, "repeat": rep, "fold": k,
                         "r2_log": r2_score(va["y"], p)})
    print(f"  part 1 repeat {rep + 1}/{N_REPEATS} done")

r1 = pd.DataFrame(rows, columns=["part", "config", "repeat", "fold", "r2_log"])
if RUN_PART1:
    ps = r1.pivot_table(index=["repeat", "fold"], columns="config", values="r2_log")
    s1 = []
    for c in CONFIGS:
        dlt = (ps[c] - ps["EXP_ONLY"]).values
        s1.append({"config": c, "r2_mean": ps[c].mean(),
                   "delta_vs_EXP": dlt.mean(), "delta_p2.5": np.percentile(dlt, 2.5),
                   "delta_p97.5": np.percentile(dlt, 97.5),
                   "frac_splits_better": (dlt > 0).mean()})
    print("\n--- PART 1: across papers, XGBoost, 5×5 GroupKFold by DOI, paired vs EXP_ONLY ---")
    print(pd.DataFrame(s1).round(3).to_string(index=False))
    print("  (p2.5–p97.5 = spread over re-splits of one dataset, not a confidence interval)")
else:
    print("PART 1 skipped (RUN_PART1 = False); results from the previous run stand.")

# ===========================================================================
# PART 2 — within papers, strict blocks (re-test of NB14, base_compound key)
# ===========================================================================
EMPTY = {"", "nan", "none", "null", "-", "n/a", "na", "undoped", "pristine", "pure", "0"}
def present(s):
    s = s.where(s.notna(), "")
    return ~s.astype(str).str.strip().str.lower().isin(EMPTY)

w = master[is_use_h2(master["use_h2"]) & master["y"].notna() &
           master["doi_key"].notna() & master["base_compound"].notna()].copy()
w["eg"] = pd.to_numeric(w["bandgap_eV"], errors="coerce")
w["base_compound"] = w["base_compound"].astype(str).str.strip()
w["modified"] = False
for c in ["dopant_paper", "secondary_phase_paper"]:
    if c in w.columns:
        w["modified"] |= present(w[c])
BLOCK_COND = [c for c in ["run_type", "scavenger_class", "co_catalyst_type",
                          "co_catalyst_loading", "cutoff_nm_final", "lamp_power_W",
                          "catalyst_mass_g", "volume_mL", "organic_vol_fraction",
                          "T_reaction_C"] if c in X_ex.columns]
# master also carries columns with these names (run_type, scavenger_class, ...);
# drop them so the merge does not create *_x / *_y suffixes — X_exp_v2 is the
# single source of block conditions, exactly as in NB13/NB14.
w = w.drop(columns=[c for c in BLOCK_COND + v2 if c in w.columns])
w = w.merge(X_ex[["source_row_id"] + BLOCK_COND], on="source_row_id", how="left") \
     .merge(X_v2[["source_row_id"] + v2], on="source_row_id", how="left")

def cond_str(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.map(lambda v: "na" if pd.isna(v) else f"{v:.4g}")
    return s.astype(str).fillna("na")

w["block"] = w["doi_key"].astype(str)
for c in BLOCK_COND:
    w["block"] = w["block"] + "|" + cond_str(w[c])

def n_elements(f):
    return len(set(re.findall(r"[A-Z][a-z]?", str(f))))

agg = (w.groupby(["block", "base_compound"])
         .agg(doi_key=("doi_key", "first"), y=("y", "mean"), eg=("eg", "mean"),
              modified=("modified", "max"), **{c: (c, "mean") for c in v2})
         .reset_index())
agg["n_el"] = agg["base_compound"].map(n_elements)
agg = agg[agg.groupby("block")["base_compound"].transform("nunique") >= 2].reset_index(drop=True)

pairs = []
for b, idx in agg.groupby("block").indices.items():
    for a in range(len(idx)):
        for c in range(a + 1, len(idx)):
            i, j = idx[a], idx[c]
            dy = agg.at[j, "y"] - agg.at[i, "y"]
            if abs(dy) < PAIR_MIN:
                continue
            mi, mj = bool(agg.at[i, "modified"]), bool(agg.at[j, "modified"])
            pairs.append({"i": i, "j": j, "dy": dy, "block": b,
                          "type": "mixed" if mi != mj else
                                  ("both_modified" if mi else "both_pristine"),
                          "same_nel": agg.at[i, "n_el"] == agg.at[j, "n_el"]})
pairs = pd.DataFrame(pairs)
SUBSETS = {"all": pairs,
           "mixed": pairs[pairs["type"] == "mixed"],
           "non_mixed": pairs[pairs["type"] != "mixed"],
           "equal_complexity": pairs[(pairs["type"] != "mixed") & pairs["same_nel"]]}
print(f"\nPART 2 — strict blocks {agg['block'].nunique()}, DOIs {agg['doi_key'].nunique()}, "
      f"materials {len(agg)} | pairs: " + ", ".join(f"{k} {len(v)}" for k, v in SUBSETS.items()))

FEATS = ["eg"] + v2
def demean(v, blocks):
    v = pd.DataFrame(v)
    return (v - v.groupby(np.asarray(blocks)).transform("mean")).values

def oof(yvals, rep):
    adois = np.array(sorted(agg["doi_key"].unique()))
    perm = np.random.default_rng(SEED + rep).permutation(adois)
    f_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    f = agg["doi_key"].map(f_of).values
    pred = np.full(len(agg), np.nan)
    for k in range(N_FOLDS):
        tr, va = np.where(f != k)[0], np.where(f == k)[0]
        if len(va) == 0:
            continue
        imp = SimpleImputer(strategy="median").fit(agg.iloc[tr][FEATS].values.astype(float))
        Xtr = demean(imp.transform(agg.iloc[tr][FEATS].values.astype(float)), agg.iloc[tr]["block"])
        Xva = demean(imp.transform(agg.iloc[va][FEATS].values.astype(float)), agg.iloc[va]["block"])
        ytr = demean(yvals[tr], agg.iloc[tr]["block"]).ravel()
        p = xgb_model(depth=3, n=300).fit(Xtr, ytr).predict(Xva)
        pred[va] = demean(p, agg.iloc[va]["block"]).ravel()
    return pred

def acc(pred, sub):
    d_ = pred[sub["j"].values] - pred[sub["i"].values]
    return np.where(d_ == 0, 0.5, (np.sign(d_) == np.sign(sub["dy"].values)).astype(float)).mean()

yv = agg["y"].values
accs = {k: [] for k in SUBSETS}
for rep in range(N_REPEATS):
    p = oof(yv, rep)
    for k, sub in SUBSETS.items():
        if len(sub):
            accs[k].append(acc(p, sub))
print("  XGBoost OOF pair accuracy (mean of 5 repeats): " +
      " | ".join(f"{k} {np.mean(v):.3f}" for k, v in accs.items() if v))
print("  NB14 reference (anion-blind descriptors): all 0.643 | mixed 0.789 | "
      "non_mixed 0.605 | equal_complexity 0.542")

perm_rows = []
test_sets = {k: SUBSETS[k] for k in ["non_mixed", "equal_complexity"] if len(SUBSETS[k]) >= 20}
if test_sets:
    p0 = oof(yv, 0)
    obs = {k: acc(p0, s) for k, s in test_sets.items()}
    rng = np.random.default_rng(SEED + 2000)
    bidx = list(agg.groupby("block").indices.values())
    null = {k: [] for k in test_sets}
    for _ in range(N_PERM):
        yp = yv.copy()
        for idx in bidx:
            yp[idx] = yv[rng.permutation(idx)]
        pp = oof(yp, 0)
        for k, s in test_sets.items():
            null[k].append(acc(pp, s))
    print(f"\n  Permutation test (repeat 0, {N_PERM} permutations):")
    for k, s in test_sets.items():
        nl = np.array(null[k])
        pv = (1 + np.sum(nl >= obs[k])) / (1 + N_PERM)
        print(f"    {k:17s} observed {obs[k]:.3f} | null mean {nl.mean():.3f}, p95 "
              f"{np.percentile(nl, 95):.3f} | p = {pv:.3f} | n_pairs {len(s)}, "
              f"blocks {s['block'].nunique()}")
        perm_rows.append({"part": 2, "config": f"perm_{k}", "repeat": 0, "fold": -1,
                          "r2_log": np.nan, "pair_acc": obs[k], "p_value": pv})
    print("  Note: 2 tests here + NB14's earlier tests — treat isolated 0.01 < p < 0.05 "
          "as exploratory.")

out = pd.concat([r1.assign(pair_acc=np.nan, p_value=np.nan),
                 pd.DataFrame([{"part": 2, "config": f"oof_{k}", "repeat": -1, "fold": -1,
                                "r2_log": np.nan, "pair_acc": np.mean(v), "p_value": np.nan}
                               for k, v in accs.items() if v]),
                 pd.DataFrame(perm_rows)], ignore_index=True)
out_tbl = f"{SCHEMA}.activity_retest_nb18" + ("" if RUN_PART1 else "_part2")
spark.createDataFrame(out.replace({np.nan: None})).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(out_tbl)
print(f"\n✓ Written: {out_tbl} ({len(out)} rows)")
print("\n=== NB18 COMPLETE — exploratory phase closed; next: consolidation ===")
